# Aggregation

In [1]:
import os 
os.chdir("Data Sets")

In [3]:
import pandas as pd
df = pd.read_csv("all_athlete_games.csv", low_memory=False)
sample = df[(df["Year"]==2020) & (df["Medal"].notna())][["Name","Gender","Age","NOC","Sport","Medal"]].reset_index(drop=True)
print(sample.shape)
print(sample.head(5))

(2449, 6)
                    Name  Gender   Age  NOC              Sport   Medal
0              ABALO Luc    Male  36.0  FRA           Handball    Gold
1          ABBOTT Monica  Female  36.0  USA  Baseball/Softball  Silver
2       ABDELAZIZ Feryal  Female  22.0  EGY             Karate    Gold
3            ABDI Bashir    Male  32.0  BEL          Athletics  Bronze
4  ABDULLAH Rahmat Erwin    Male  20.0  INA      Weightlifting  Bronze


In [5]:
# GroupBy - Split-Apply-Combine, Grouping by multiple columns

# The idea in three steps: Split the data into groups based on a column's values -> 
# apply a calculation to each group seperately -> combine the results back into one table.

# .groupby automates the entire process in one call

# syntax - df.groupby(column_or_list)

# grouping by a single column

print(sample.groupby("NOC").size().sort_values(ascending=False).head(8))

# size counts how many rows fall into each group.
# This is the split by NOC and combine (count per group) happening automatically

NOC
USA    298
ROC    149
FRA    141
GBR    141
CHN    141
AUS    131
JPN    131
CAN     85
dtype: int64


In [7]:
print(sample.groupby("Sport")["Age"].mean().sort_values(ascending=False))

Sport
Equestrian               42.395349
Sailing                  30.444444
Cycling Road             30.250000
Shooting                 30.222222
Handball                 30.053763
Fencing                  29.477778
Baseball/Softball        29.239316
Canoe Slalom             28.666667
Volleyball               28.388889
Modern Pentathlon        28.333333
Golf                     28.333333
Beach Volleyball         28.250000
Triathlon                28.095238
Canoe Sprint             28.000000
Water Polo               27.897436
Basketball               27.888889
Tennis                   27.833333
Cycling Mountain Bike    27.666667
Karate                   27.625000
Hockey                   27.481481
Rowing                   27.458333
Wrestling                27.277778
3x3 Basketball           27.000000
Rugby Sevens             26.653846
Boxing                   26.634615
Judo                     26.480392
Athletics                26.473913
Table Tennis             26.433333
Cycling Track 

In [8]:
# groupby multiple columns

print(sample.groupby(["NOC","Medal"]).size())

NOC  Medal 
ARG  Bronze     25
     Silver     18
ARM  Bronze      2
     Silver      2
AUS  Bronze     67
              ... 
USA  Silver    110
UZB  Bronze      2
     Gold        3
VEN  Gold        1
     Silver      3
Length: 210, dtype: int64


In [10]:
# umstack() - pivots the second grouping level from rows into columns
print(sample.groupby(["NOC","Medal"]).size().unstack(fill_value=0))

Medal  Bronze  Gold  Silver
NOC                        
ARG        25     0      18
ARM         2     0       2
AUS        67    36      28
AUT         5     1       1
AZE         4     0       3
..        ...   ...     ...
UGA         1     2       1
UKR        21     1       7
USA        75   113     110
UZB         2     3       0
VEN         0     1       3

[93 rows x 3 columns]


In [11]:
# Aggregations 
# .agg({'col1':'sum,'col2':mean})

# syntax -

# df.groupby(col).agg(mapping)
# here mapping can be a dict, list or a single function name

# Different aggregation per column using a dict

sample.groupby("NOC").agg({"Age":"mean","Name":"count"})

,Age,Name
NOC,,
ARG,27.116279,43
ARM,27.500000,4
AUS,26.725191,131
AUT,28.428571,7
AZE,30.428571,7
...,...,...
UGA,22.500000,4
UKR,25.793103,29
USA,26.607383,298


In [14]:
# multiple aggregations on the same column

sample.groupby("Sport")["Age"].agg(["mean","min","max","count"])

,mean,min,max,count
Sport,,,,
3x3 Basketball,27.000000,19.0,35.0,24
Archery,25.766667,17.0,39.0,30
Artistic Gymnastics,22.819672,16.0,30.0,61
Artistic Swimming,26.100000,19.0,35.0,30
Athletics,26.473913,17.0,38.0,230
Badminton,25.750000,23.0,33.0,24
Baseball/Softball,29.239316,20.0,43.0,117
Basketball,27.888889,20.0,40.0,72
Beach Volleyball,28.250000,24.0,39.0,12


In [15]:
# combining both

sample.groupby("NOC").agg({"Age":["mean","std"],"Medal":"count"})

Age           Medal
          mean       std count
NOC                           
ARG  27.116279  4.499785    43
ARM  27.500000  2.380476     4
AUS  26.725191  7.004453   131
AUT  28.428571  2.507133     7
AZE  30.428571  3.952094     7
..         ...       ...   ...
UGA  22.500000  1.914854     4
UKR  25.793103  5.595785    29
USA  26.607383  5.603349   298
UZB  24.400000  4.774935     5
VEN  26.750000  6.448514     4

[93 rows x 3 columns]

| .agg() input                              | Result                                              |
|--------------------------------------------------|--------------------------------------------------------------|
| .agg("mean")                                          | One function applied to the whole grouped Series/column          |
| .agg(["mean","min","max"])                              | Multiple functions, same column                                     |
| .agg({"col1":"sum", "col2":"mean"})                       | Different function per column                                          |
| .agg({"col1": ["mean","std"], "col2":"count"})              | Mix of single and multiple functions per column                            |